In [25]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

ruta_actual = os.getcwd()
ruta_base = os.path.dirname(ruta_actual) if "notebooks" in ruta_actual else ruta_actual
ruta_csv = os.path.join(ruta_base, "data", "processed", "autos_limpios.csv")

df = pd.read_csv(ruta_csv)
df.head()

,titulo,precio,ano,kilometraje,marca
0,Hyundai Accent,2712895.0,2012.0,205058.0,Hyundai
1,TOYOTA RAV4,9546276.0,2023.0,49149.0,Toyota
2,Hyundai Tucson IMPECABLE,2500000.0,2013.0,204087.0,Hyundai
3,Nissan Qashqai,2500000.0,2011.0,247650.0,Nissan
4,Hyundai Accent,13704045.0,2021.0,71120.0,Hyundai


In [26]:
#Definir variables de entrada y salida
X = df[['ano', 'kilometraje', 'marca']]
y = df['precio']

print("Características de entrada (X):")
print(X.head())
print("\nVariable objetivo (y):")
print(y.head())

Características de entrada (X):
      ano  kilometraje    marca
0  2012.0     205058.0  Hyundai
1  2023.0      49149.0   Toyota
2  2013.0     204087.0  Hyundai
3  2011.0     247650.0   Nissan
4  2021.0      71120.0  Hyundai

Variable objetivo (y):
0     2712895.0
1     9546276.0
2     2500000.0
3     2500000.0
4    13704045.0
Name: precio, dtype: float64


In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Datos de entrenamiento: {X_train.shape[0]} filas")
print(f"Datos de prueba: {X_test.shape[0]} filas")

Datos de entrenamiento: 628 filas
Datos de prueba: 158 filas


In [28]:
#Definir cómo tratar la columna categórica (marca)
transformador_categorico = OneHotEncoder(handle_unknown='ignore')

#Aplicar la transformación a columnas de texto
preprocesador = ColumnTransformer(
    transformers=[
        ('cat', transformador_categorico, ['marca'])
    ],
    remainder='passthrough'
)

#Crear el Pipeline completo usando el algoritmo RandomForest
pipeline_rf = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', RandomForestRegressor(n_estimators=100, random_state=42))
])

#Entrenar el modelo
pipeline_rf.fit(X_train, y_train)
print("Modelo RandomForest entrenado con éxito")

Modelo RandomForest entrenado con éxito


In [29]:
#Realizar las predicciones
predicciones = pipeline_rf.predict(X_test)

#Calcular métricas cuantitativas
mae = mean_absolute_error(y_test, predicciones)
r2 = r2_score(y_test, predicciones)

print("=== EVALUACIÓN DEL MODELO ===")
print(f"Error Absoluto Medio (MAE): ${mae:,.0f}".replace(",", "."))
print(f"Coeficiente de Determinación (R²): {r2:.2f}")
print("=============================")

=== EVALUACIÓN DEL MODELO ===
Error Absoluto Medio (MAE): $2.173.849
Coeficiente de Determinación (R²): 0.63


In [30]:
#Ejemplo de Tasación
mi_auto = pd.DataFrame([{
    'ano': 2018,
    'kilometraje': 85000,
    'marca': 'Toyota'
}])

precio_estimado = pipeline_rf.predict(mi_auto)[0]

print(f"Tasación para un Toyota año 2018 con 85.000 Km:")
print(f"Precio estimado por el modelo: ${precio_estimado:,.0f}".replace(",", "."))

Tasación para un Toyota año 2018 con 85.000 Km:
Precio estimado por el modelo: $9.066.842
